# 00 · Descarga e inventario de Polyvore

Descarga el dataset de Han et al. (2017), repositorio [`xthan/polyvore-dataset`](https://github.com/xthan/polyvore-dataset) (licencia Apache 2.0). Solo metadatos en JSON, sin imágenes.

**Salida:** `data/external/polyvore/` (no se sube a git; este notebook la regenera).

> Cita: Han, X., Wu, Z., Jiang, Y.-G. y Davis, L. S. (2017). *Learning Fashion Compatibility with Bidirectional LSTMs*. ACM Multimedia.

In [1]:
import hashlib
import json
import re
import tarfile
import urllib.request
from collections import Counter
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("No se encontró data/raw. Abre el notebook dentro del repositorio.")

DESTINO = ROOT / "data" / "external" / "polyvore"
DESTINO.mkdir(parents=True, exist_ok=True)
BASE = "https://github.com/xthan/polyvore-dataset/raw/master/"
ARCHIVOS = ["polyvore.tar.gz", "category_id.txt", "LICENSE"]
SHA256_TAR = "37c36efc67cabdaa58f2d73cbc1c5143d87f9918ff3d4da20e271e40c89e902b"
CONTENIDO = ["train_no_dup.json", "valid_no_dup.json", "test_no_dup.json",
             "fill_in_blank_test.json", "fashion_compatibility_prediction.txt"]

## 1. Descarga y verificación

In [2]:
for nombre in ARCHIVOS:
    ruta = DESTINO / nombre
    if not ruta.exists():
        urllib.request.urlretrieve(BASE + nombre, ruta)
        print("Descargado:", nombre)

sha = hashlib.sha256((DESTINO / "polyvore.tar.gz").read_bytes()).hexdigest()
if sha != SHA256_TAR:
    raise ValueError(f"polyvore.tar.gz no coincide con el archivo verificado (sha256 {sha}).")

with tarfile.open(DESTINO / "polyvore.tar.gz") as tar:
    nombres = tar.getnames()
    if sorted(nombres) != sorted(CONTENIDO):
        raise ValueError(f"Contenido inesperado en el tar: {nombres}")
    for nombre in CONTENIDO:
        if not (DESTINO / nombre).exists():
            tar.extract(nombre, DESTINO, filter="data")
print("OK:", sorted(p.name for p in DESTINO.iterdir()))

Descargado: polyvore.tar.gz


Descargado: category_id.txt


Descargado: LICENSE


OK: ['LICENSE', 'category_id.txt', 'fashion_compatibility_prediction.txt', 'fill_in_blank_test.json', 'polyvore.tar.gz', 'test_no_dup.json', 'train_no_dup.json', 'valid_no_dup.json']


## 2. Inventario

In [3]:
def tid(item):
    """Id del producto en Polyvore, sacado de la URL de su imagen."""
    return re.search(r"tid=(\d+)", item["image"]).group(1)

particiones = {s: json.loads((DESTINO / f"{s}_no_dup.json").read_text(encoding="utf-8"))
               for s in ["train", "valid", "test"]}
categorias = dict(
    linea.split(" ", 1) for linea in (DESTINO / "category_id.txt").read_text(encoding="utf-8").splitlines() if linea.strip()
)

filas, productos = [], {}
for s, outfits in particiones.items():
    items = [it for o in outfits for it in o["items"]]
    for it in items:
        productos.setdefault(tid(it), set()).add(s)
    filas.append({
        "particion": s,
        "outfits": len(outfits),
        "items": len(items),
        "productos_unicos": len({tid(it) for it in items}),
        "items_sin_nombre": sum(not it["name"].strip() for it in items),
    })
inventario = pd.DataFrame(filas)
display(inventario)
print("Productos que aparecen en más de una partición oficial:",
      sum(len(v) > 1 for v in productos.values()), "de", len(productos))

,particion,outfits,items,productos_unicos,items_sin_nombre
0,train,17316,114806,84497,1317
1,valid,1497,9070,8923,202
2,test,3076,18604,18169,352


Productos que aparecen en más de una partición oficial: 0 de 111589


In [4]:
largo = Counter(len(o["items"]) for outfits in particiones.values() for o in outfits)
display(pd.Series(largo, name="outfits").sort_index().rename_axis("items por outfit").to_frame())

cat_uso = Counter(str(it["categoryid"]) for outfits in particiones.values() for o in outfits for it in o["items"])
top = pd.DataFrame(cat_uso.most_common(25), columns=["categoryid", "items"])
top["nombre"] = top["categoryid"].map(categorias)
print(f"Categorías usadas: {len(cat_uso)} (de {len(categorias)} en category_id.txt). Las 25 más frecuentes:")
display(top)

sin_nombre = sorted(set(cat_uso) - set(categorias))
print("categoryid sin nombre en category_id.txt:", sin_nombre)

,outfits
items por outfit,
4,2400
5,3691
6,4028
7,3903
8,7867


Categorías usadas: 380 (de 380 en category_id.txt). Las 25 más frecuentes:


,categoryid,items,nombre
0,64,5118,Earrings
1,37,5098,Shoulder Bags
2,62,4719,Necklaces
3,11,4226,Tops
4,43,4113,Pumps
5,46,4101,Sandals
6,57,4042,Sunglasses
7,261,3904,Ankle Booties
8,106,3540,Bracelets & Bangles
9,38,3459,Clutches


categoryid sin nombre en category_id.txt: []


In [5]:
ejemplo = particiones["train"][0]
print(ejemplo["name"], "|", ejemplo["desc"])
display(pd.DataFrame(ejemplo["items"])[["index", "name", "categoryid", "price", "likes"]]
        .assign(categoria=lambda t: t["categoryid"].astype(str).map(categorias)))

Casual | A fashion look from January 2017 by beebeely-look featuring Fuji, Citizens of Humanity, casual, casualoutfit, Packandgo, winterjacket and gearbest


,index,name,categoryid,price,likes,categoria
0,1,mock neck embroidery suede sweatshirt,4495,24.0,10,Sweatshirts
1,2,luxe double zip hooded jacket,25,150.0,2250,Jackets
2,3,citizens humanity high rise rocket hem jean,27,248.0,2437,Jeans
3,4,suede tie short boots,261,37.0,2,Ankle Booties
4,5,cloth travel school backpack,259,22.0,2,Backpacks
5,6,,1967,160.0,3,Tech Accessories
6,7,polyvore,2,-1.0,51,Clothing
